# 06 - AWS Orchestration And Monitoring

This notebook documents how the project extends from notebook execution into production-style AWS orchestration using Lambda/EventBridge, EMR, Redshift, and CloudWatch.

## Production Flow

```text
S3 file arrival or EventBridge schedule
  -> Lambda function
  -> EMR Spark job / EMR step
  -> S3 Bronze/Silver/Gold outputs
  -> Redshift Spectrum or COPY into Redshift
  -> CloudWatch logs and alarms
```

## Lambda Trigger Template

The repository includes `lambda/lambda_submit_emr_step.py`. Lambda should submit the EMR job. It should not run Spark directly.

In [ ]:
lambda_template = """
import os
import boto3

emr = boto3.client('emr')

def lambda_handler(event, context):
    cluster_id = os.environ['EMR_CLUSTER_ID']
    code_bucket = os.environ['CODE_BUCKET']
    return emr.add_job_flow_steps(
        JobFlowId=cluster_id,
        Steps=[{
            'Name': 'LoanShield PySpark Pipeline',
            'ActionOnFailure': 'CONTINUE',
            'HadoopJarStep': {
                'Jar': 'command-runner.jar',
                'Args': ['spark-submit', f's3://{code_bucket}/loanshield/jobs/main_pipeline.py']
            }
        }]
    )
"""
print(lambda_template)

## CloudWatch Monitoring Plan

Track the following operational signals:

- EMR step success or failure
- Driver and executor logs
- Pipeline duration
- Bronze, Silver, Gold, and rejected record counts
- Data quality score
- Rejected-record spike alerts

## Redshift Analytics Plan

For portfolio implementation, Redshift can consume Gold data using either:

- Redshift Spectrum external tables over the S3 Gold path
- COPY command into a managed Redshift table

Spectrum is a natural fit because S3 remains the data lake source of truth.